# 介護データ標準化チャレンジ — scorer notebook (Carnets / iPad)

Self-contained version of `care_score/` for the 2026 医療データハッカソン.
Reads 14 days of care records per user, scores the 8 items, and writes the
2-week submission CSV.

**Setup on iPad (Carnets Plus)**

1. In the Files app, copy this notebook **and the `data/` folder** (from the repo) into
   *On My iPad ▸ Carnets*. The layout must be:
   ```
   care_hackathon_carnets.ipynb
   data/sample_3users/*.csv
   data/test_7users/*.csv
   ```
2. Open the notebook in Carnets and choose **Run ▸ Run All Cells**.
3. Outputs are written to a `submission/` folder next to the notebook.

Only NumPy and pandas are needed; both ship with Carnets Plus (pandas is optional and
only used for nicer tables).

In [ ]:
import csv, json, re, sys
from collections import defaultdict
from pathlib import Path
import numpy as np
try:
    import pandas as pd
except ImportError:  # plain Carnets without pandas
    pd = None

# ---- paths (edit if your files live elsewhere) --------------------------------
DATA_DIR = Path("data")
SAMPLE_GT_RECORDS = DATA_DIR / "sample_3users/care_hackathon_ground_truth_records_3users.csv"
SAMPLE_GT_2WEEK   = DATA_DIR / "sample_3users/care_hackathon_2week_summaries_ground_truth_3users.csv"
SAMPLE_RECORDS    = DATA_DIR / "sample_3users/care_hackathon_records_3users_14days.csv"
TEST_RECORDS      = DATA_DIR / "test_7users/care_hackathon_student_records_7users_14days.csv"
TEST_SUMMARIES    = DATA_DIR / "test_7users/care_hackathon_2week_summaries_7users.csv"
OUT_DIR           = Path("submission")

print("python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__ if pd else "not available")
for p in (SAMPLE_GT_RECORDS, TEST_RECORDS, TEST_SUMMARIES):
    print(("found   " if p.exists() else "MISSING ") + str(p))

## 1. Lexicon

Every daily record is a 「。」-separated list of template sentences. Slots 1–6 are always
食事 → 入浴 → 運動 → 排泄 → 睡眠 → 認知・意欲; optional trailing sentences give 介助負担, a
リスク note, and 「注意点として、…がみられる」. Each sentence maps to one score.
Entries derived from the 3-user sample are integers; hand-mapped test-only sentences may be
fractional (expected value) where the tier is ambiguous.

The embedded table is the hand-mapped version. Drop a regenerated `lexicon.json` next to the notebook to use LLM-scored entries instead.

In [ ]:
LEXICON = {
"食事は8割程度摂取": [
"食事",
4.0
],
"入浴介助あり": [
"入浴",
3.0
],
"午後のレクリエーションに参加": [
"運動",
3.0
],
"排泄は見守りで可能": [
"排泄",
4.0
],
"夜間は良眠": [
"睡眠",
5.0
],
"レクリエーションに参加": [
"認知_意欲",
4.0
],
"日常動作に介助が必要な場面が多い": [
"介助負担",
4.0
],
"経過は概ね安定": [
"リスク",
1.0
],
"食事は少量のみ": [
"食事",
2.0
],
"声かけを行ったが入浴には至らなかった": [
"入浴",
1.0
],
"体操参加を拒否": [
"運動",
1.0
],
"排泄は一部介助": [
"排泄",
3.0
],
"夜間覚醒は2回": [
"睡眠",
3.0
],
"表情はやや硬い": [
"認知_意欲",
2.0
],
"拒否があり対応に時間を要した": [
"介助負担",
5.0
],
"食欲低下や水分摂取不足に注意": [
"リスク",
4.0
],
"昼食は全量摂取": [
"食事",
5.0
],
"移乗と洗体に介助を要した": [
"入浴",
3.0
],
"衣服操作に一部介助を要した": [
"排泄",
3.0
],
"一度目覚めたが再入眠できた": [
"睡眠",
4.0
],
"声かけに対する反応は良好": [
"認知_意欲",
4.0
],
"複数場面で介助を要した": [
"介助負担",
4.0
],
"主食は半量程度、副食は少し残した": [
"食事",
3.0
],
"体調を考慮し部分清拭とした": [
"入浴",
2.0
],
"短時間のみレクリエーションに参加": [
"運動",
2.0
],
"トイレ誘導後は見守りのみ": [
"排泄",
4.0
],
"夜間に複数回目覚めた": [
"睡眠",
3.0
],
"軽度の混乱はあるが指示理解可能": [
"認知_意欲",
3.0
],
"経過観察が必要": [
"リスク",
3.0
],
"少し残したが大部分は摂取": [
"食事",
4.0
],
"歩行訓練15分を行った": [
"運動",
4.0
],
"夜間覚醒なく休めていた": [
"睡眠",
5.0
],
"介助下で入浴を実施": [
"入浴",
3.0
],
"声かけと見守りで排泄できた": [
"排泄",
4.0
],
"軽い中途覚醒あり": [
"睡眠",
4.0
],
"摂取量は普段よりやや少なめ": [
"食事",
3.0
],
"散歩は5分程度で終了": [
"運動",
2.0
],
"トイレ動作の一部に介助が必要": [
"排泄",
3.0
],
"夜間の眠りが浅く日中の眠気が強い": [
"睡眠",
2.0
],
"入浴または排泄で介助が必要": [
"介助負担",
4.0
],
"主食・副食ともに残さず摂取": [
"食事",
5.0
],
"歩行訓練10分実施": [
"運動",
3.0
],
"睡眠状態は良好": [
"睡眠",
5.0
],
"日中は落ち着いて過ごした": [
"認知_意欲",
4.0
],
"声かけしたが摂取量は少なかった": [
"食事",
2.0
],
"運動は実施せず": [
"運動",
1.0
],
"意欲低下あり": [
"認知_意欲",
2.0
],
"夜間覚醒があり注意が必要": [
"リスク",
4.0
],
"摂取量は概ね良好": [
"食事",
4.0
],
"短時間の体操は問題なく実施": [
"運動",
3.0
],
"朝食は完食": [
"食事",
5.0
],
"傾眠や軽度の混乱に注意": [
"リスク",
3.0
],
"活動への参加はみられなかった": [
"運動",
1.0
],
"排泄介助あり": [
"排泄",
2.0
],
"日中傾眠が目立つ": [
"睡眠",
2.0
],
"ふらつきがあり常時見守りを要した": [
"介助負担",
5.0
],
"夜間覚醒は1回": [
"睡眠",
4.0
],
"表情は穏やか": [
"認知_意欲",
4.0
],
"上肢体操に参加": [
"運動",
3.0
],
"軽度の疲労感はあるが大きな問題なし": [
"リスク",
1.0
],
"声かけのみで入浴できた": [
"入浴",
5.0
],
"午前の体操と午後の散歩に参加": [
"運動",
5.0
],
"排泄は自立": [
"排泄",
5.0
],
"会話は活発": [
"認知_意欲",
5.0
],
"介助負担は少ない": [
"介助負担",
0.0
],
"特記事項なし": [
"リスク",
0.0
],
"食事は十分量を摂取できている": [
"食事",
5.0
],
"浴室内の動作は自立していた": [
"入浴",
5.0
],
"体操に概ね参加できた": [
"運動",
4.0
],
"入浴は自立して実施": [
"入浴",
5.0
],
"歩行訓練20分実施": [
"運動",
5.0
],
"意欲的に参加": [
"認知_意欲",
5.0
],
"排泄は介助なく実施": [
"排泄",
5.0
],
"表情明るく反応良好": [
"認知_意欲",
5.0
],
"散歩15分実施": [
"運動",
4.0
],
"散歩20分を安定して行った": [
"運動",
5.0
],
"見守り不要で実施できた": [
"介助負担",
0.0
],
"状態は安定": [
"リスク",
0.0
],
"レクリエーション後も疲労少なく過ごした": [
"運動",
4.0
],
"他者との交流もみられた": [
"認知_意欲",
5.0
],
"トイレ動作は自立": [
"排泄",
5.0
],
"排泄動作は介助下で実施": [
"排泄",
2.0
],
"促しにより短時間参加できた": [
"認知_意欲",
3.0
],
"主食・副食ともにほぼ食べられていた": [
"食事",
4.0
],
"本日は清拭のみ実施": [
"入浴",
2.0
],
"活動への参加は消極的": [
"認知_意欲",
2.0
],
"声かけにより軽い運動を実施": [
"運動",
2.0
],
"声かけで参加": [
"認知_意欲",
3.0
],
"夜間覚醒後の様子を確認": [
"リスク",
3.0
],
"6割程度の摂取にとどまった": [
"食事",
3.0
],
"疲労感があり本日の歩行訓練は中止": [
"運動",
1.0
],
"服薬拒否があり確認が必要": [
"リスク",
5.0
],
"トイレ移乗に介助を要した": [
"排泄",
2.0
],
"食欲低下あり": [
"食事",
2.0
],
"会話は少なめ": [
"認知_意欲",
2.0
],
"食事は半分程度摂取": [
"食事",
3.0
],
"見守り下で活動に加われた": [
"認知_意欲",
3.0
],
"数口から3割程度の摂取にとどまる": [
"食事",
1
],
"一部介助で入浴実施": [
"入浴",
4
],
"洗体時のみ介助を要した": [
"入浴",
4
],
"見守りと一部介助で入浴を終えた": [
"入浴",
4
],
"入浴は行わず清拭で対応": [
"入浴",
2
],
"集団体操に最後まで参加": [
"運動",
5
],
"車椅子でホールまで移動し活動を見学": [
"運動",
1.5
],
"睡眠がやや浅い": [
"睡眠",
3.5
],
"夜間覚醒が3回以上あり": [
"睡眠",
1.5
],
"混乱があり介助困難": [
"介助負担",
5
],
"必要時の声かけ程度で過ごせた": [
"介助負担",
1
],
"軽い確認のみで対応可能": [
"介助負担",
1
],
"便秘傾向と睡眠状態に注意": [
"リスク",
4
],
"ふらつきがあり転倒に注意": [
"リスク",
4.5
],
"強い拒否がみられ重点的な対応が必要": [
"リスク",
5
]
}

print(len(LEXICON), 'sentences')
from collections import Counter
print(Counter(d for d, _ in LEXICON.values()))

# If a lexicon.json (e.g. regenerated with care_score/llm_lexicon.py) sits next to this notebook, prefer it.
_lex_file = Path("lexicon.json")
if _lex_file.exists():
    LEXICON = {s: [v["domain"], v["score"]] for s, v in json.loads(_lex_file.read_text(encoding="utf-8")).items()}
    print("loaded", len(LEXICON), "sentences from lexicon.json;", sum(1 for v in json.loads(_lex_file.read_text(encoding="utf-8")).values() if v.get("source") == "llm"), "from LLM")

## 2. Scoring rules

In [ ]:
DOMAINS = ["食事", "入浴", "運動", "排泄", "睡眠", "認知_意欲"]
ALL = DOMAINS + ["介助負担", "リスク"]
SCORE_COL = {k: f"{k}_score" for k in ALL}
MEAN_COL = {k: f"{k}_score_平均" for k in ALL}
SIMPLE_COL = dict(zip(ALL, ["食事", "入浴", "運動・歩行", "排泄自立度", "睡眠状態", "認知・意欲", "介助負担", "リスク"]))

# flag -> (score key, comparator, threshold); reproduces 注意すべき変化_正解 exactly on the sample
FLAGS = [
    ("食事低下", "食事", "<=", 2),
    ("入浴拒否・困難", "入浴", "<=", 1),
    ("活動低下", "運動", "<=", 1),
    ("睡眠不良", "睡眠", "<=", 2),
    ("意欲低下・混乱", "認知_意欲", "<=", 2),
    ("介助負担大", "介助負担", ">=", 4),
    ("高リスク", "リスク", ">=", 4),
]
FLAG_RE = re.compile(r"^注意点として、(.+)がみられる$")

def split_sentences(text):
    return [s for s in text.split("。") if s]

def fit_fallback(gt_rows):
    """介助負担 / リスク ~ six domain scores (least squares on the sample ground truth)."""
    X = np.array([[float(r[SCORE_COL[d]]) for d in DOMAINS] for r in gt_rows])
    A = np.c_[X, np.ones(len(X))]
    return {k: np.linalg.lstsq(A, np.array([float(r[SCORE_COL[k]]) for r in gt_rows]), rcond=None)[0]
            for k in ("介助負担", "リスク")}

# weights fitted on the 3-user sample, used if the sample GT file is not present
FALLBACK_W = {
    "介助負担": np.array([0.243591, -1.254963, 0.024826, -0.619542, 0.039691, -0.000933, 8.095956]),
    "リスク":   np.array([-0.123337, -0.466592, 0.078802, -0.15073, -0.526331, -0.232862, 7.088162]),
}

def score_record(text, w, warnings):
    sents = split_sentences(text)
    scores, flags = {}, None
    for i, s in enumerate(sents):
        if i == 0:
            continue                      # opening sentence carries no score
        m = FLAG_RE.match(s)
        if m:
            flags = {f.strip() for f in m.group(1).split("、")}
            continue
        entry = LEXICON.get(s)
        if entry is None:
            dom = DOMAINS[i - 1] if 1 <= i <= 6 else None
            warnings.append(f"unknown sentence (slot {i}, assumed {dom}): {s}")
            if dom:
                scores[dom] = 3.0         # neutral guess
            continue
        scores[entry[0]] = float(entry[1])
    for d in DOMAINS:
        scores.setdefault(d, 3.0)
    x = np.array([scores[d] for d in DOMAINS] + [1.0])
    if "介助負担" not in scores:           # 介助負担 >= 4 always has a sentence -> clip to 0..3
        scores["介助負担"] = float(np.clip(x @ w["介助負担"], 0, 3))
    if "リスク" not in scores:             # リスク >= 3 always has a note -> clip to 0..2
        scores["リスク"] = float(np.clip(x @ w["リスク"], 0, 2))
    if flags is not None:                 # 注意点 sentence appears only when リスク >= 4
        scores["リスク"] = max(scores["リスク"], 4.0)
        for name, key, op, thr in FLAGS:
            if name in flags:
                scores[key] = min(scores[key], thr) if op == "<=" else max(scores[key], thr)
            else:
                scores[key] = max(scores[key], thr + 1) if op == "<=" else min(scores[key], thr - 1)
    return scores

def derive_flags(scores):
    out = []
    for name, key, op, thr in FLAGS:
        v = round(scores[key])
        if (op == "<=" and v <= thr) or (op == ">=" and v >= thr):
            out.append(name)
    return " / ".join(out)

def predict(records, w):
    warnings, daily = [], []
    for r in records:
        sc = score_record(r["記録内容"], w, warnings)
        row = {"利用者ID": r["利用者ID"], "日付": r["日付"], "記録内容": r["記録内容"]}
        row.update({SCORE_COL[k]: sc[k] for k in ALL})
        row["注意すべき変化"] = derive_flags(sc)
        daily.append(row)
    return daily, warnings

def aggregate(daily):
    by_user = defaultdict(list)
    for r in daily:
        by_user[r["利用者ID"]].append(r)
    agg = {}
    for uid, rows in sorted(by_user.items()):
        a = {MEAN_COL[k]: round(float(np.mean([r[SCORE_COL[k]] for r in rows])), 2) for k in ALL}
        a["注意日数"] = sum(1 for r in rows if r["注意すべき変化"])
        a["期間開始"], a["期間終了"] = min(r["日付"] for r in rows), max(r["日付"] for r in rows)
        agg[uid] = a
    return agg

def fmt(v):
    return str(int(v)) if float(v).is_integer() else f"{v:.2f}"

def read_csv(p):
    with open(p, encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))

def show(rows, cols):
    if pd is not None:
        from IPython.display import display
        display(pd.DataFrame([[r[c] for c in cols] for r in rows], columns=cols))
    else:
        print(",".join(cols))
        for r in rows:
            print(",".join(str(r[c]) for c in cols))
print("ok")

## 3. Fit the 介助負担 / リスク fallback and check against the sample

If `data/sample_3users/` is present the regression is refit and the whole pipeline is checked
against the ground truth. Expected: the six domain scores match exactly, overall 2-week MAE ≈ 0.01.

In [ ]:
if SAMPLE_GT_RECORDS.exists():
    gt = read_csv(SAMPLE_GT_RECORDS)
    W = fit_fallback(gt)
    print("fallback regression refit on", len(gt), "sample records")

    bad = sum(derive_flags({k: float(r[SCORE_COL[k]]) for k in ALL}) != r["注意すべき変化_正解"] for r in gt)
    print(f"flag thresholds reproduce 注意すべき変化_正解: {len(gt)-bad}/{len(gt)}")

    daily_s, warn = predict(gt, W)
    for m in warn: print("WARNING:", m)
    mae_daily = {k: float(np.mean([abs(p[SCORE_COL[k]] - float(g[SCORE_COL[k]])) for p, g in zip(daily_s, gt)])) for k in ALL}
    print("daily MAE:", {k: round(v, 3) for k, v in mae_daily.items()}, "| overall", round(np.mean(list(mae_daily.values())), 4))

    if SAMPLE_GT_2WEEK.exists():
        gt2 = {r["利用者ID"]: r for r in read_csv(SAMPLE_GT_2WEEK)}
        agg_s = aggregate(daily_s)
        mae2 = {k: float(np.mean([abs(agg_s[u][MEAN_COL[k]] - float(gt2[u][MEAN_COL[k]])) for u in agg_s])) for k in ALL}
        print("2-week MAE:", {k: round(v, 3) for k, v in mae2.items()}, "| overall", round(np.mean(list(mae2.values())), 4))
        rows = []
        for u in agg_s:
            rows.append(dict({"利用者ID": u, "source": "pred"}, **{SIMPLE_COL[k]: fmt(agg_s[u][MEAN_COL[k]]) for k in ALL}, 注意日数=agg_s[u]["注意日数"]))
            rows.append(dict({"利用者ID": u, "source": "truth"}, **{SIMPLE_COL[k]: gt2[u][MEAN_COL[k]] for k in ALL}, 注意日数=gt2[u]["注意日数"]))
        show(rows, ["利用者ID", "source"] + [SIMPLE_COL[k] for k in ALL] + ["注意日数"])
else:
    W = FALLBACK_W
    print("sample ground truth not found -> using embedded regression weights")

## 4. Score the 7 test users and write the submission

In [ ]:
records = read_csv(TEST_RECORDS)
summaries = {r["利用者ID"]: r for r in read_csv(TEST_SUMMARIES)} if TEST_SUMMARIES.exists() else {}
daily, warnings = predict(records, W)
for m in warnings: print("WARNING:", m)
agg = aggregate(daily)
print(len(records), "records,", len(agg), "users")

OUT_DIR.mkdir(parents=True, exist_ok=True)

# (a) 2-week means: same columns as care_hackathon_2week_summaries_ground_truth_3users.csv  -> upload this
p_2week = OUT_DIR / "care_hackathon_7users_2week_prediction.csv"
cols = ["利用者ID", "期間開始", "期間終了", "利用者特徴", "2週間サマリー"] + [MEAN_COL[k] for k in ALL] + ["注意日数"]
with open(p_2week, "w", encoding="utf-8", newline="") as f:
    wr = csv.writer(f); wr.writerow(cols)
    for uid, a in agg.items():
        s = summaries.get(uid, {})
        wr.writerow([uid, s.get("期間開始", a["期間開始"]), s.get("期間終了", a["期間終了"]), s.get("利用者特徴", ""), s.get("2週間サマリー", "")]
                    + [fmt(a[MEAN_COL[k]]) for k in ALL] + [a["注意日数"]])

# (b) daily scores: same columns as care_hackathon_ground_truth_records_3users.csv
p_daily = OUT_DIR / "care_hackathon_7users_records_prediction.csv"
cols = ["利用者ID", "日付", "利用者特徴", "記録内容"] + [SCORE_COL[k] for k in ALL] + ["注意すべき変化_正解"]
with open(p_daily, "w", encoding="utf-8", newline="") as f:
    wr = csv.writer(f); wr.writerow(cols)
    for r in daily:
        wr.writerow([r["利用者ID"], r["日付"], summaries.get(r["利用者ID"], {}).get("利用者特徴", ""), r["記録内容"]]
                    + [fmt(r[SCORE_COL[k]]) for k in ALL] + [r["注意すべき変化"]])

# (c) simple layout from the hackathon page
p_simple = OUT_DIR / "care_hackathon_7users_simple.csv"
with open(p_simple, "w", encoding="utf-8", newline="") as f:
    wr = csv.writer(f); wr.writerow(["user_id"] + [SIMPLE_COL[k] for k in ALL])
    for uid, a in agg.items():
        wr.writerow([uid] + [fmt(a[MEAN_COL[k]]) for k in ALL])

for p in (p_2week, p_daily, p_simple): print("wrote", p)
show([dict({"user_id": u}, **{SIMPLE_COL[k]: fmt(a[MEAN_COL[k]]) for k in ALL}, 注意日数=a["注意日数"]) for u, a in agg.items()],
     ["user_id"] + [SIMPLE_COL[k] for k in ALL] + ["注意日数"])

## 5. Daily detail (optional)

In [ ]:
show([dict({k: r[k] for k in ("利用者ID", "日付")}, **{SIMPLE_COL[k]: fmt(r[SCORE_COL[k]]) for k in ALL}, 注意すべき変化=r["注意すべき変化"]) for r in daily],
     ["利用者ID", "日付"] + [SIMPLE_COL[k] for k in ALL] + ["注意すべき変化"])